# Демонстрация модели

Лучшая модель — large_moe_v16k_data2_148m:
- MoE (от Mixtral): 8 экспертов, 2 активных
- 321M параметров, из них 118M активных
- обучена с нуля на 148M токенов чувашского текста (корпус v2)
- 7 часов на M4 Pro

In [1]:
import json

import torch
import pandas as pd
from transformers import AutoModelForCausalLM, PreTrainedTokenizerFast


device = 'mps' if torch.backends.mps.is_available() else 'cpu'
checkpoint = 'checkpoints/large_moe_v16k_data2_148m'

tokenizer = PreTrainedTokenizerFast.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device).eval()

print(type(model).__name__)
print(f'{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M параметров')
print(f'слоев: {model.config.num_hidden_layers}, d_model: {model.config.hidden_size}, контекст: {model.config.max_position_embeddings}')

/Users/leoqay/homework/text_processings/hw1/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1026 22:37:38.142000 4641 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.



Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]


Loading weights:  76%|███████▌  | 56/74 [00:00<00:00, 554.46it/s]


Loading weights: 100%|██████████| 74/74 [00:00<00:00, 628.09it/s]

MixtralForCausalLM
320.6M параметров
слоев: 8, d_model: 1024, контекст: 512


In [2]:
torch.manual_seed(0)


def generate(prompt, n=80, greedy=False):
    ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)

    if greedy:
        params = dict(do_sample=False, repetition_penalty=1.3)
    else:
        params = dict(do_sample=True, top_p=0.9, temperature=0.7, repetition_penalty=1.15)

    out = model.generate(
        ids,
        max_new_tokens=n,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        **params,
    )

    return tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

## 1. Продолжение текста

Начала фраз на разные темы: новости, школа, природа, история, быт.

In [ ]:
prompts = {
    'Столица Чувашской Республики…': 'Чӑваш Республикин тӗп хули',
    'В нашей деревне…': 'Пирӗн ялта',
    'Летом дети…': 'Ҫулла ачасем',
    'Константин Иванов…': 'Константин Иванов',
    'Сегодня в школе…': 'Паян шкулта',
    'Чувашский язык…': 'Чӑваш чӗлхи',
    'Зимой…': 'Хӗлле',
    'Моя мама…': 'Манӑн анне',
    'Я люблю...': 'Эпӗ юрататӑп',
}

for translation, prompt in prompts.items():
    print(f'### {translation}')
    print(prompt + generate(prompt, greedy=False))
    print()

### Столица Чувашской Республики…


Чӑваш Республикин тӗп хули ятне, Чӑваш Енӗн мухтавлӑ ывӑлӗ, Совет Союзӗн Геройӗ Николай Гаврилов (17.10.1943 ҫур.) тивӗҫнӗ.
Левентейпе Пачман пӗр-пӗрне юратнине каласа парас килчӗ унӑн.

### В нашей деревне…


Пирӗн ялта та ҫакӑн пек пысӑк, пысӑк ӗҫ туса ирттерес тесе эпир пурте пӗрле тӑрӑшатпӑр.
Вӗсем те ман пекех: «Кавруҫ, Сӑпани!» — теҫҫӗ.
Вӑл мана хӑйӗн юратнӑ юррине юрлама сӗнчӗ.
Ҫакнашкал йӑла-йӗркепе ҫыхӑннӑ историсене те аса илчӗ вӑл.
— Ун ҫинчен мӗн калама пултаратӑр?
Вӑл, хӑйне хӑй алла илсе, малалла утрӗ.
— Ҫук

### Летом дети…


Ҫулла ачасем урамра вылянӑ.
Кинотеатрта ӑна пурте паллаҫҫӗ: «Пиллӗкмӗш класра вӗреннӗ чухне эпӗ Ксюшӑна куртӑм, вӑл мана та палларӗ».
Ӑна мӗнле майпа туртса кӑларнине пӗлмен пирки Ковшовпа Мышкин пӗр-пӗринпе калаҫса татӑлаҫҫӗ те, ӑна пӗр ӗҫ хушаҫҫӗ.
— Мӗне кирлӗ мана сан шутпа? — ыйтрӗ вӑл.
Машина патне

### Константин Иванов…


Константин Иванов, Юрий Сементер поэт-писатель тата ыттисем те.
Тупӑшӑва Чӑваш Республикин культура министрӗ Вадим Ефимов хутшӑннӑ, вӑл чӑваш халӑхӗн йӑли-йӗркипе ҫыхӑннӑ чылай ыйтӑва хускатнине палӑртнӑ.
— Ҫук! — терӗҫ хӗрарӑмсем, — пӗр сӑмах та каламаҫҫӗ.
Анчах эпир хамӑр та, ҫамрӑксем, тӑван чӗлхесемпе калаҫатпӑр, тӑван чӗлхепе калаҫмастпӑр, тӑван чӗлхепе калаҫмастпӑр…
— Вӑл та

### Сегодня в школе…


Паян шкулта ачасене чӑваш чӗлхине вӗрентме пулӑшакан «Чӑваш чӗлхи урокӗ» иртнӗ.
«Чӑваш чӗлхине вӗренмелли кӗнеке» номинацире ҫӗнтернисем ҫаксем пулчӗҫ:
1. Чӑваш Республикин вӗренӳ институчӗн чӑваш чӗлхипе литератури кафедрин преподавателӗ Н.П.Нагорнов,
2. Шупашкарти педагогика университечӗн чӑваш чӗлхипе литератури кафедрин доценчӗ А.А.Рекеев,
3. Шупашкар хулинчи чӑваш чӗлхипе литературин вӗрентекенӗсем В.И

### Чувашский язык…


Чӑваш чӗлхи — чӑваш чӗлхи.
Вӑл хӑйӗн пурнӑҫне Иеговӑна халалланӑ чухне , ҫав ӗҫе туса тӑма пултарсан , унӑн ӑна тума май та пулман .
ПӖТЕТКЕНСЕМ ХУНӐ ТУСА ПАРНЕСА ҪИНЧЕН КАЛАСА ИЛМЕЛЛИ СӖНЕСӖР? | «Канаш» — Чӗмпӗр чӑвашӗсен хаҫачӗ
ҪЫР

### Зимой…


Хӗллехи сивӗсем те питӗ лайӑх.
Сӗтел хушшине ларсах, сӗтел хушшинче ларакансен сӑмахӗсене тӑнларӗҫ:
Чупса ҫитрӗҫ хайхисем.
— Кам ку? — ыйтрӗ Павлушран.
Вӑл мана хӑй патне илсе каясшӑнччӗ, анчах ман патӑма пӗр ҫын килсе тухрӗ.

### Моя мама…


Манӑн анне ҫук.
«Тупнӑ чуста, — тет вӑл, ман ҫине пӑхса.
Вӗсем мӗн пирки ӗмӗтленнине, мӗн шухӑшланине пӗлес килчӗ.

### Я люблю...


Эпӗ юрататӑп! — тесе кӑшкӑрса ячӗ вӑл.
Аякра, сӑрт ҫинче, кӑвак пӗлӗт курӑннӑ.
— Ку вӑл пӗр кӗтмен ҫӗртен пулнӑ япала!
— Ахалех ҫав, — терӗ Дзержинский, унтан вӑл хӑйӗн сӑмахне тепӗр хут каларӗ:
— Эпӗ сире каларӑм, эсир мана хирӗҫ чӗнмерӗр.
— Мӗншӗн? — тӗлӗнсе ыйтрӗ Базаров, ҫак вӑхӑтра хӑй ӑшӗнче кулса илчӗ



## 2. Вопросы с перефразировками

Модель базовая, без инструкций, поэтому вопрос задаю двумя способами:
- как начало утверждения: "Столица Чувашии - …"
- в формате вопрос-ответ: "Ыйту: … Хурав:"

Каждый факт спрашиваю тремя формулировками.

Если модель знает факт, ответ должен быть одинаковым во всех трех.

Декодирование жадное.

In [4]:
items = json.load(open('chuvash_llm/data/benchmark_chv.json'))

rows = []

for item in items[:10]:
    for prompt in item['prompts']:
        answer = generate(prompt, n=12, greedy=True).split('\n')[0]
        rows.append({'факт': item['fact'], 'вопрос': prompt, 'ответ модели': answer})

pd.set_option('display.max_colwidth', 80)
pd.DataFrame(rows)

,факт,вопрос,ответ модели
0,Столица Чувашии — Чебоксары,Чӑваш Енӗн тӗп хули —,Шупашкар.
1,Столица Чувашии — Чебоксары,Чӑваш Республикин тӗп хули —,Шупашкар.
2,Столица Чувашии — Чебоксары,Ыйту: Чӑваш Енӗн тӗп хули мӗнле хула? Хурав:,Шупашкар хулин Мускав районӗ
3,Столица России — Москва,Раҫҫейӗн тӗп хули —,Мускав.
4,Столица России — Москва,Раҫҫей Федерацийӗн тӗп хули —,Мускав.
5,Столица России — Москва,Ыйту: Раҫҫейӗн тӗп хули мӗнле хула? Хурав:,Шупашкар
6,Столица Татарстана — Казань,Тутарстанӑн тӗп хули —,Хусан.
7,Столица Татарстана — Казань,Тутарстан Республикин тӗп хули —,Тутарстан Республики.
8,Столица Татарстана — Казань,Ыйту: Тутарстанӑн тӗп хули мӗнле хула? Хурав:,"Шупашкар, Ҫӗнӗ Шупашкар."
9,Поэму «Нарспи» написал Константин Иванов,«Нарспи» поэмӑна ҫырнӑ чӑваш поэчӗ —,Чӑваш Республикин тава тивӗҫлӗ вӗрентекенӗ


## 3. Бенчмарк

32 факта (география, литература, история, календарь, здравый смысл) × 3 формулировки × 4 варианта ответа.
Получается всего 96 вопросов с 4 вариантами ответа.

Ответ модели - вариант с наибольшей вероятностью:
- сырая оценка log P(вариант | вопрос) любит короткие и частые слова:
    - модель со случайными весами всегда выбирает самый короткий вариант
- поэтому из нее вычитаю log P(вариант) без вопроса

Метрики:
- acc_calib — доля правильных ответов с такой поправкой (главная метрика)
- acc, acc_norm — без поправки и с делением на длину варианта
- consistency — доля фактов, где все 3 формулировки дают один ответ
- consistent_correct — доля фактов, где все 3 ответа правильные

In [5]:
results = json.load(open('results/benchmark.json'))
random = json.load(open('results/random_controls.json'))['acc_calib']

print(f'acc_calib у случайных моделей: от {min(random)} до {max(random)}')

table = {
    name: {'bits per char': r['bpc'], **r['summary']}
    for name, r in results.items()
}

pd.DataFrame(table).T

acc_calib у случайных моделей: от 0.198 до 0.385


,bits per char,acc,acc_norm,acc_calib,consistency,consistent_correct
large_moe_v16k_data2_148m,1.1496,0.490,0.479,0.438,0.375,0.188
xl_v16k_data2,1.1550,0.510,0.375,0.438,0.469,0.219
large_v16k_data2,1.1625,0.500,0.479,0.458,0.312,0.125
large_moe_v16k_data2,1.1766,0.490,0.406,0.490,0.531,0.312
medium_v16k,1.2448,0.396,0.354,0.375,0.531,0.250
small_v16k,1.2649,0.365,0.406,0.365,0.531,0.188
tiny_v16k,1.3599,0.281,0.281,0.260,0.375,0.094
tiny_v8k,1.3801,0.333,0.344,0.323,0.469,0.156
tiny_v16k_20m,1.5150,0.302,0.281,0.312,0.531,0.156
tiny_moe_v16k_20m,1.4932,0.281,0.260,0.292,0.625,0.156


Ответы лучшей модели:

In [6]:
answers = [
    {'факт': item['fact'], 'ответы на 3 формулировки': ' / '.join(item['answers']), 'все верны': item['all_correct']}
    for item in results['large_moe_v16k_data2_148m']['items']
]

pd.DataFrame(answers)

,факт,ответы на 3 формулировки,все верны
0,Столица Чувашии — Чебоксары,Шупашкар / Шупашкар / Шупашкар,True
1,Столица России — Москва,Мускав / Мускав / Шупашкар,False
2,Столица Татарстана — Казань,Хусан / Хусан / Шупашкар,False
3,Поэму «Нарспи» написал Константин Иванов,Константин Иванов / Константин Иванов / Константин Иванов,True
4,«Евгения Онегина» написал Пушкин,Александр Пушкин / Александр Пушкин / Константин Иванов,False
5,«Войну и мир» написал Лев Толстой,Александр Пушкин / Александр Пушкин / Константин Иванов,False
6,Чувашский космонавт — Андриян Николаев,Андриян Николаев / Андриян Николаев / Константин Иванов,False
7,Первый космонавт мира — Юрий Гагарин,Юрий Гагарин / Юрий Гагарин / Юрий Гагарин,True
8,Создатель чувашского алфавита — Иван Яковлев,Иван Яковлев / Иван Яковлев / Иван Яковлев,True
9,Чапаев — легендарный командир Гражданской войны,Юрий Гагарин / Юрий Гагарин / Юрий Гагарин,False
